# 04 - Prototype

## PB-06: Prototipo interactivo para stakeholders

Este prototipo interactivo resume y explora el dataset de phishing a partir del EDA realizado previamente.

### Funciones
- ver un resumen ejecutivo del dataset
- comparar el análisis con y sin duplicados
- explorar variables relevantes para la clasificación
- revisar el comportamiento de las variables por clase
- identificar rápidamente las variables más asociadas con `Result`

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import ipywidgets as widgets
from IPython.display import display, Markdown

In [2]:
df = pd.read_csv("../data/raw/dataset.csv")
df_nodup = df.drop_duplicates().copy()

df["Result_label"] = df["Result"].map({
    -1: "Phishing",
     1: "Legítimo"
})

df_nodup["Result_label"] = df_nodup["Result"].map({
    -1: "Phishing",
     1: "Legítimo"
})

df.head()

,having_IP_Address,URL_Length,Shortining_Service,having_At_Symbol,double_slash_redirecting,Prefix_Suffix,having_Sub_Domain,SSLfinal_State,Domain_registeration_length,Favicon,...,Iframe,age_of_domain,DNSRecord,web_traffic,Page_Rank,Google_Index,Links_pointing_to_page,Statistical_report,Result,Result_label
0,-1,1,1,1,-1,-1,-1,-1,-1,1,...,1,-1,-1,-1,-1,1,1,-1,-1,Phishing
1,1,1,1,1,1,-1,0,1,-1,1,...,1,-1,-1,0,-1,1,1,1,-1,Phishing
2,1,0,1,1,1,-1,-1,-1,-1,1,...,1,1,-1,1,-1,1,0,-1,-1,Phishing
3,1,0,1,1,1,-1,-1,-1,1,1,...,1,-1,-1,1,-1,1,-1,1,-1,Phishing
4,1,0,-1,1,1,-1,1,1,-1,1,...,1,-1,-1,0,-1,1,1,1,1,Legítimo


In [3]:
important_vars = [
    "SSLfinal_State",
    "URL_of_Anchor",
    "Prefix_Suffix",
    "web_traffic",
    "having_Sub_Domain",
    "Request_URL",
    "Links_in_tags",
    "SFH",
    "Domain_registeration_length",
    "Google_Index"
]

pretty_names = {
    "SSLfinal_State": "Estado final SSL",
    "URL_of_Anchor": "Comportamiento de enlaces ancla",
    "Prefix_Suffix": "Uso de prefijo/sufijo en la URL",
    "web_traffic": "Nivel de tráfico web",
    "having_Sub_Domain": "Presencia de subdominios",
    "Request_URL": "Comportamiento de URLs solicitadas",
    "Links_in_tags": "Enlaces dentro de etiquetas",
    "SFH": "Server Form Handler (SFH)",
    "Domain_registeration_length": "Duración de registro del dominio",
    "Google_Index": "Indexación en Google"
}

null_count = int(df.isnull().sum().sum())
duplicate_count = int(df.duplicated().sum())
n_rows, n_cols = df.shape
n_rows_nodup = len(df_nodup)

target_counts = df["Result_label"].value_counts()
target_pct = (df["Result_label"].value_counts(normalize=True) * 100).round(2)

corr_with_target = df.corr(numeric_only=True)["Result"].drop("Result")
corr_abs_sorted = corr_with_target.abs().sort_values(ascending=False)

In [4]:
kpi_html = widgets.HTML(
    value=f"""
    <div style="display:flex; gap:12px; flex-wrap:wrap; margin:10px 0;">
        <div style="padding:12px 16px; border:1px solid #ddd; border-radius:10px; min-width:170px;">
            <b>Registros</b><br>{n_rows}
        </div>
        <div style="padding:12px 16px; border:1px solid #ddd; border-radius:10px; min-width:170px;">
            <b>Variables</b><br>{n_cols}
        </div>
        <div style="padding:12px 16px; border:1px solid #ddd; border-radius:10px; min-width:170px;">
            <b>Nulos</b><br>{null_count}
        </div>
        <div style="padding:12px 16px; border:1px solid #ddd; border-radius:10px; min-width:170px;">
            <b>Duplicados</b><br>{duplicate_count}
        </div>
        <div style="padding:12px 16px; border:1px solid #ddd; border-radius:10px; min-width:170px;">
            <b>Phishing</b><br>{target_pct.get('Phishing', 0)}%
        </div>
        <div style="padding:12px 16px; border:1px solid #ddd; border-radius:10px; min-width:170px;">
            <b>Legítimo</b><br>{target_pct.get('Legítimo', 0)}%
        </div>
    </div>
    """
)

display(Markdown("## Resumen ejecutivo del dataset"))
display(kpi_html)

## Resumen ejecutivo del dataset

HTML(value='\n    <div style="display:flex; gap:12px; flex-wrap:wrap; margin:10px 0;">\n        <div style="pa…

In [5]:
variable_dropdown = widgets.Dropdown(
    options=[(pretty_names[v], v) for v in important_vars],
    value="SSLfinal_State",
    description="Variable:"
)

usar_sin_duplicados = widgets.Checkbox(
    value=False,
    description="Usar sin duplicados"
)

filtrar_clase = widgets.Checkbox(
    value=False,
    description="Filtrar clase"
)

clase_dropdown = widgets.Dropdown(
    options=[("Phishing", "Phishing"), ("Legítimo", "Legítimo")],
    value="Phishing",
    description="Clase:"
)

vista_radio = widgets.RadioButtons(
    options=["Frecuencia general", "Frecuencia por clase", "Porcentaje por clase"],
    value="Frecuencia por clase",
    description="Vista:"
)

top_n_slider = widgets.IntSlider(
    value=8,
    min=5,
    max=15,
    step=1,
    description="Top N:"
)

usar_valor_absoluto = widgets.Checkbox(
    value=True,
    description="Ordenar por valor absoluto"
)

In [6]:
def render_variable(variable, usar_sin_duplicados, filtrar_clase, clase, vista):
    data = df_nodup.copy() if usar_sin_duplicados else df.copy()

    if filtrar_clase:
        data = data[data["Result_label"] == clase]

    if data.empty:
        display(Markdown("**No hay datos disponibles con esa combinación de filtros.**"))
        return

    corr_value = corr_with_target[variable]

    if corr_value > 0:
        corr_text = "Más asociada a sitios legítimos"
    else:
        corr_text = "Más asociada a sitios phishing"

    display(Markdown(
        f"""
### Explorador de variable

- **Variable técnica:** `{variable}`
- **Variable amigable:** **{pretty_names[variable]}**
- **Correlación con `Result`:** **{corr_value:.3f}**
- **Interpretación general:** **{corr_text}**
- **Dataset usado:** **{'sin duplicados' if usar_sin_duplicados else 'completo'}**
- **Registros visibles:** **{len(data)}**
"""
    ))

    order = sorted(data[variable].dropna().unique())
    fig, ax = plt.subplots(figsize=(10, 5))

    if vista == "Frecuencia general":
        sns.countplot(x=variable, data=data, order=order, ax=ax)
        ax.set_title(f"Frecuencia de {pretty_names[variable]}")
        ax.set_xlabel(pretty_names[variable])
        ax.set_ylabel("Conteo")

    elif vista == "Frecuencia por clase":
        if filtrar_clase:
            sns.countplot(x=variable, data=data, order=order, ax=ax)
            ax.set_title(f"{pretty_names[variable]} | Clase: {clase}")
            ax.set_xlabel(pretty_names[variable])
            ax.set_ylabel("Conteo")
        else:
            tabla = pd.crosstab(data[variable], data["Result_label"]).sort_index()
            tabla = tabla.reindex(columns=["Phishing", "Legítimo"], fill_value=0)
            tabla.plot(kind="bar", stacked=True, ax=ax)
            ax.set_title(f"{pretty_names[variable]} por clase objetivo")
            ax.set_xlabel(pretty_names[variable])
            ax.set_ylabel("Conteo")
            ax.legend(title="Clase")
            plt.subplots_adjust(bottom=0.22, left=0.10, right=0.98, top=0.88)
            plt.show()
            display(tabla)
            return

    else:  # Porcentaje por clase
        if filtrar_clase:
            tabla_pct = (data[variable].value_counts(normalize=True).sort_index() * 100).round(2)
            tabla_pct.plot(kind="bar", ax=ax)
            ax.set_title(f"Porcentaje de {pretty_names[variable]} | Clase: {clase}")
            ax.set_xlabel(pretty_names[variable])
            ax.set_ylabel("Porcentaje")
            plt.subplots_adjust(bottom=0.22, left=0.10, right=0.98, top=0.88)
            plt.show()
            display(tabla_pct.to_frame("percentage"))
            return
        else:
            tabla_pct = (
                pd.crosstab(data[variable], data["Result_label"], normalize="columns") * 100
            ).sort_index().round(2)
            tabla_pct = tabla_pct.reindex(columns=["Phishing", "Legítimo"], fill_value=0)
            tabla_pct.plot(kind="bar", stacked=True, ax=ax)
            ax.set_title(f"Porcentaje de {pretty_names[variable]} por clase objetivo")
            ax.set_xlabel(pretty_names[variable])
            ax.set_ylabel("Porcentaje")
            ax.legend(title="Clase")
            plt.subplots_adjust(bottom=0.22, left=0.10, right=0.98, top=0.88)
            plt.show()
            display(tabla_pct)
            return

    plt.subplots_adjust(bottom=0.22, left=0.10, right=0.98, top=0.88)
    plt.show()

In [7]:
def render_correlations(top_n, usar_valor_absoluto):
    if usar_valor_absoluto:
        vars_top = corr_abs_sorted.head(top_n).index
        serie = corr_with_target.loc[vars_top]
        titulo = f"Top {top_n} variables más relacionadas con Result"
    else:
        serie = corr_with_target.sort_values(ascending=False).head(top_n)
        titulo = f"Top {top_n} variables con mayor correlación positiva con Result"

    colors = ["#2E86AB" if v > 0 else "#D1495B" for v in serie.values]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh([pretty_names.get(v, v) for v in serie.index], serie.values, color=colors)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_title(titulo)
    ax.set_xlabel("Correlación")
    ax.set_ylabel("Variable")
    plt.tight_layout()
    plt.show()

    interpretable = serie.to_frame("correlation")
    interpretable["interpretation"] = interpretable["correlation"].apply(
        lambda x: "Más asociada a Legítimo" if x > 0 else "Más asociada a Phishing"
    )
    display(interpretable)

In [8]:
display(Markdown("## Explorador interactivo de variables"))

controls_left = widgets.VBox([
    variable_dropdown,
    usar_sin_duplicados,
    filtrar_clase,
    clase_dropdown,
    vista_radio
])

output_left = widgets.interactive_output(
    render_variable,
    {
        "variable": variable_dropdown,
        "usar_sin_duplicados": usar_sin_duplicados,
        "filtrar_clase": filtrar_clase,
        "clase": clase_dropdown,
        "vista": vista_radio
    }
)

display(controls_left, output_left)

## Explorador interactivo de variables

Output(outputs=({'output_type': 'display_data', 'data': {'text/plain': '<IPython.core.display.Markdown object>…

In [9]:
display(Markdown("## Explorador interactivo de correlaciones"))

controls_right = widgets.VBox([
    top_n_slider,
    usar_valor_absoluto
])

output_right = widgets.interactive_output(
    render_correlations,
    {
        "top_n": top_n_slider,
        "usar_valor_absoluto": usar_valor_absoluto
    }
)

display(controls_right, output_right)

## Explorador interactivo de correlaciones

Output()

## Interpretación final

Este prototipo busca ser útil para stakeholders porque:

- resume el estado del dataset en una vista ejecutiva
- incorpora el hallazgo de duplicados
- permite explorar las variables más relevantes para `Result`
- compara la distribución por clase
- muestra de forma clara qué variables tienen mayor relación con la variable objetivo

Esta notebook corresponde al backlog item **PB-06: Prototipo interactivo con ipywidgets**.